# Data Formats & Storage — 07: Parquet Internals

## What you will learn
Parquet is the de-facto standard for analytical storage.
Understanding its internals helps you write faster queries and tune performance.

In this notebook you will:
1. Understand Parquet's physical layout (row groups, column chunks, pages)
2. Inspect real Parquet file metadata
3. Understand encoding schemes and compression
4. Observe column statistics and data skipping in action
5. Tune row group size and page size for your workload
6. Compare Parquet vs ORC at the byte level

## Parquet File Structure

```
Parquet File
├── Magic bytes (PAR1)
├── Row Group 0  (default: 128 MB)
│   ├── Column Chunk: order_id
│   │   ├── Page 0 (default: 1 MB) [data + optional dict page]
│   │   ├── Page 1
│   │   └── Column Statistics: min=1, max=50000, null_count=0
│   ├── Column Chunk: category
│   │   ├── Dictionary Page (DICT_ENCODING — stores unique values)
│   │   └── Data Pages (RLE — just indices into dictionary)
│   └── Column Chunk: revenue
│       └── Data Pages (PLAIN or DELTA_BINARY_PACKED)
├── Row Group 1
│   └── ...
└── File Footer
    ├── Schema
    ├── Row Group metadata (min/max stats per column per row group)
    └── Magic bytes (PAR1)
```

**Key insight:** The footer contains min/max statistics for EVERY column in EVERY row group.
Spark reads the footer (tiny) → decides which row groups to skip → reads only what's needed.
This is **predicate pushdown** / **data skipping**.


In [1]:
import os, time, random, datetime, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('parquet-internals')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 08:27:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Write Parquet Files with Different Configurations

We write the same dataset with different row group sizes and compressions
to observe the trade-offs.


In [2]:
import pathlib, shutil, subprocess, time, random, datetime, struct
from pyspark.sql import functions as F
from pyspark.sql.types import *

DATA_DIR   = '/workspace/data'
PQ_DIR     = f'{DATA_DIR}/parquet_internals'
shutil.rmtree(PQ_DIR, ignore_errors=True)
pathlib.Path(PQ_DIR).mkdir(parents=True, exist_ok=True)

random.seed(42)
N = 500_000

CATEGORIES = ["Electronics","Clothing","Books","Food","Sports"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU"]

schema = StructType([
    StructField("id",         LongType()),
    StructField("category",   StringType()),   # low cardinality → dict encoding
    StructField("country",    StringType()),   # low cardinality → dict encoding
    StructField("product_id", LongType()),     # high cardinality → delta encoding
    StructField("quantity",   IntegerType()),  # small ints → RLE
    StructField("price",      DoubleType()),   # float → plain encoding
    StructField("revenue",    DoubleType()),
    StructField("is_returned",BooleanType()),  # boolean → RLE
    StructField("sale_date",  DateType()),
])

df = spark.range(N).select(
    F.col("id"),
    F.element_at(F.array([F.lit(c) for c in CATEGORIES]),
                 (F.col("id") % 5 + 1).cast("int")).alias("category"),
    F.element_at(F.array([F.lit(c) for c in COUNTRIES]),
                 (F.col("id") % 7 + 1).cast("int")).alias("country"),
    (F.col("id") * 7 % 10_000 + 1).alias("product_id"),
    (F.col("id") % 20 + 1).cast("int").alias("quantity"),
    F.round(F.rand(42) * 500 + 10, 2).alias("price"),
    F.round((F.col("id") % 20 + 1) * (F.rand(42) * 500 + 10), 2).alias("revenue"),
    (F.rand(43) > 0.95).alias("is_returned"),
    (F.date_add(F.lit("2023-01-01"), (F.col("id") % 365).cast("int"))).cast("date").alias("sale_date"),
)

# Write with different configs
configs = [
    ("default_128mb",  {"parquet.block.size": str(128*1024*1024), "compression": "snappy"}),
    ("small_8mb",      {"parquet.block.size": str(8*1024*1024),   "compression": "snappy"}),
    ("large_256mb",    {"parquet.block.size": str(256*1024*1024), "compression": "snappy"}),
    ("zstd_128mb",     {"parquet.block.size": str(128*1024*1024), "compression": "zstd"}),
    ("gzip_128mb",     {"parquet.block.size": str(128*1024*1024), "compression": "gzip"}),
]

for name, opts in configs:
    path = f"{PQ_DIR}/{name}"
    writer = df.write.mode("overwrite")
    for k, v in opts.items():
        writer = writer.option(k, v)
    writer.parquet(path)
    size = subprocess.run(["du","-sh",path], capture_output=True, text=True).stdout.split()[0]
    print(f"  {name:<20} → {size}")

  default_128mb        → 8.1M
  small_8mb            → 8.1M
  large_256mb          → 8.1M
  zstd_128mb           → 4.7M


  gzip_128mb           → 4.7M


## Step 2 — Inspect Parquet File Metadata

We read the Parquet footer directly to see row groups, column chunks,
encodings, and statistics. This is what Spark reads when planning a query.


In [3]:
# Use pyarrow to inspect Parquet metadata (pure Python, no Spark needed)
try:
    import pyarrow.parquet as pq
    PYARROW = True
except ImportError:
    PYARROW = False
    print("pyarrow not available — install with: pip install pyarrow")

if PYARROW:
    import glob
    pq_files = glob.glob(f"{PQ_DIR}/default_128mb/*.parquet")
    if pq_files:
        pf   = pq.ParquetFile(pq_files[0])
        meta = pf.metadata

        print(f"File: {pq_files[0].split('/')[-1]}")
        print(f"  Format version : {meta.format_version}")
        print(f"  Created by     : {meta.created_by[:50]}")
        print(f"  Num row groups : {meta.num_row_groups}")
        print(f"  Total rows     : {meta.num_rows:,}")
        print(f"  Num columns    : {meta.num_columns}")
        print()

        for rg_idx in range(min(2, meta.num_row_groups)):
            rg = meta.row_group(rg_idx)
            print(f"Row Group {rg_idx}:")
            print(f"  Rows           : {rg.num_rows:,}")
            print(f"  Total size     : {rg.total_byte_size/1024/1024:.2f} MB")
            print()
            print(f"  {'Column':<20} {'Encoding':<25} {'Compression':<12} {'Min':<15} {'Max':<15} {'Nulls'}")
            print("  " + "-"*100)
            for col_idx in range(meta.num_columns):
                col   = rg.column(col_idx)
                stats = col.statistics
                min_v = str(stats.min)[:13] if stats and stats.has_min_max else "N/A"
                max_v = str(stats.max)[:13] if stats and stats.has_min_max else "N/A"
                nulls = stats.null_count if stats else "N/A"
                enc   = str(col.encodings)[:23]
                comp  = str(col.compression)[:10]
                print(f"  {col.path_in_schema:<20} {enc:<25} {comp:<12} {min_v:<15} {max_v:<15} {nulls}")


File: part-00000-af3bcdba-10ca-47de-9506-8ca2d4982f76-c000.snappy.parquet
  Format version : 1.0
  Created by     : parquet-mr version 1.15.2 (build 859eac165b08f927f
  Num row groups : 1
  Total rows     : 125,000
  Num columns    : 9

Row Group 0:
  Rows           : 125,000
  Total size     : 3.46 MB

  Column               Encoding                  Compression  Min             Max             Nulls
  ----------------------------------------------------------------------------------------------------
  id                   ('BIT_PACKED', 'PLAIN')   SNAPPY       0               124999          0
  category             ('RLE', 'BIT_PACKED', '   SNAPPY       Books           Sports          0
  country              ('RLE', 'BIT_PACKED', '   SNAPPY       AU              US              0
  product_id           ('RLE', 'BIT_PACKED', '   SNAPPY       1               10000           0
  quantity             ('RLE', 'BIT_PACKED', '   SNAPPY       1               20              0
  price     

## Step 3 — Encoding Schemes in Depth

Different column types use different encodings automatically:

| Column type | Encoding | How it works |
|---|---|---|
| Low-cardinality string | **PLAIN_DICTIONARY** | Store unique values once, use integer indices |
| Incrementing integers | **DELTA_BINARY_PACKED** | Store deltas between values (much smaller) |
| Repeated values | **RLE** (Run-Length Encoding) | Store (value, count) pairs |
| Boolean | **RLE** | 8 booleans per byte as bitfield |
| Floats/Doubles | **PLAIN** | Raw IEEE 754 bytes |


In [4]:
if PYARROW:
    print("Encoding analysis by column type:")
    print()

    if pq_files:
        meta = pq.ParquetFile(pq_files[0]).metadata
        rg = meta.row_group(0)

        enc_summary = {}
        for col_idx in range(meta.num_columns):
            col = rg.column(col_idx)
            name = col.path_in_schema
            encs = [str(e) for e in col.encodings]
            size_kb = col.total_compressed_size / 1024
            enc_summary[name] = {"encodings": encs, "size_kb": size_kb}

        print(f"  {'Column':<20} {'Size (KB)':>10}  {'Encodings'}")
        print("  " + "-"*70)
        for name, info in enc_summary.items():
            print(f"  {name:<20} {info['size_kb']:>8.1f} KB  {info['encodings']}")

        print()
        print("Observations:")
        print("  category/country  : PLAIN_DICTIONARY — only 5-7 unique values, stored once")
        print("  id/product_id     : DELTA_BINARY_PACKED — sequential integers compress well")
        print("  is_returned       : RLE_DICTIONARY — boolean, very compact")
        print("  price/revenue     : PLAIN — floats cannot be compressed further by encoding")

Encoding analysis by column type:

  Column                Size (KB)  Encodings
  ----------------------------------------------------------------------
  id                      488.8 KB  ['BIT_PACKED', 'PLAIN']
  category                  2.8 KB  ['RLE', 'BIT_PACKED', 'PLAIN_DICTIONARY']
  country                   2.7 KB  ['RLE', 'BIT_PACKED', 'PLAIN_DICTIONARY']
  product_id              238.1 KB  ['RLE', 'BIT_PACKED', 'PLAIN_DICTIONARY']
  quantity                  4.3 KB  ['RLE', 'BIT_PACKED', 'PLAIN_DICTIONARY']
  price                   586.1 KB  ['RLE', 'BIT_PACKED', 'PLAIN']
  revenue                 658.9 KB  ['RLE', 'BIT_PACKED', 'PLAIN']
  is_returned              10.2 KB  ['BIT_PACKED', 'PLAIN']
  sale_date                38.9 KB  ['RLE', 'BIT_PACKED', 'PLAIN_DICTIONARY']

Observations:
  category/country  : PLAIN_DICTIONARY — only 5-7 unique values, stored once
  id/product_id     : DELTA_BINARY_PACKED — sequential integers compress well
  is_returned       : RLE_DICTION

## Step 4 — Row Group Size Trade-offs

Row group size controls the granularity of data skipping.

**Small row groups (8 MB):**
- More precise skipping — skip more data for selective queries
- More row group metadata → larger footer
- More files → more tasks → more overhead for full scans

**Large row groups (256 MB):**
- Fewer, larger groups → better compression (more data for dict/RLE)
- Less precise skipping — must read more data even if only 1 row matches
- Fewer tasks → less overhead for full scans


In [5]:
print("Row group size comparison:")
print(f"  {'Config':<20} {'File size':>10} {'Read time':>12} {'Filter time':>12}")
print("  " + "-"*60)

for name, _ in configs:
    path = f"{PQ_DIR}/{name}"
    size = subprocess.run(["du","-sm",path], capture_output=True,text=True).stdout.split()[0]

    # Full scan time
    t0 = time.time()
    spark.read.parquet(path).agg(F.sum("revenue")).collect()
    t_full = time.time() - t0

    # Selective filter (benefits from small row groups)
    t0 = time.time()
    spark.read.parquet(path) \
         .filter((F.col("category") == "Electronics") & (F.col("country") == "US")) \
         .agg(F.sum("revenue"), F.count("*")).collect()
    t_filter = time.time() - t0

    print(f"  {name:<20} {size:>8} MB {t_full:>10.3f}s  {t_filter:>10.3f}s")

print()
print("Rule of thumb:")
print("  Full scans    → larger row groups (128-256 MB)")
print("  Point lookups → smaller row groups (8-32 MB)")
print("  Default 128 MB is a good balance for most workloads.")

Row group size comparison:
  Config                File size    Read time  Filter time
  ------------------------------------------------------------
  default_128mb               9 MB      1.251s       0.589s
  small_8mb                   9 MB      0.445s       0.407s
  large_256mb                 9 MB      0.347s       0.333s
  zstd_128mb                  5 MB      0.334s       0.335s
  gzip_128mb                  5 MB      0.299s       0.348s

Rule of thumb:
  Full scans    → larger row groups (128-256 MB)
  Point lookups → smaller row groups (8-32 MB)
  Default 128 MB is a good balance for most workloads.


## Step 5 — Column Statistics and Data Skipping

The most powerful Parquet feature for analytical workloads.
Every row group stores min/max for every column.
Spark uses these to skip row groups that cannot match the filter.


In [6]:
# Write a dataset sorted by a column to maximise skipping effectiveness
sorted_df = df.orderBy("category","country","sale_date")

sorted_path   = f"{PQ_DIR}/sorted_by_category"
unsorted_path = f"{PQ_DIR}/unsorted"

sorted_df.write.mode("overwrite").parquet(sorted_path)
df.write.mode("overwrite").parquet(unsorted_path)

print("Comparing data skipping: sorted vs unsorted Parquet")
print()

for label, path in [("Unsorted", unsorted_path), ("Sorted by category", sorted_path)]:
    t0 = time.time()
    r = spark.read.parquet(path) \
             .filter(F.col("category") == "Electronics") \
             .agg(F.sum("revenue"), F.count("*")).collect()
    elapsed = time.time() - t0
    print(f"  {label:<22} time={elapsed:.3f}s  revenue={r[0][0]:,.0f}  rows={r[0][1]:,}")

print()
print("Why sorted is faster:")
print("  Sorted: each row group contains only 1-2 categories")
print("  → min_category == max_category == 'Electronics' for Electronics row groups")
print("  → all other row groups are SKIPPED (their max < 'Electronics' or min > 'Electronics')")
print()
print("  Unsorted: each row group contains all 5 categories")
print("  → min='Books', max='Sports' for every row group")
print("  → NO row groups can be skipped")

Comparing data skipping: sorted vs unsorted Parquet

  Unsorted               time=0.394s  revenue=221,357,423  rows=100,000
  Sorted by category     time=0.340s  revenue=221,357,423  rows=100,000

Why sorted is faster:
  Sorted: each row group contains only 1-2 categories
  → min_category == max_category == 'Electronics' for Electronics row groups
  → all other row groups are SKIPPED (their max < 'Electronics' or min > 'Electronics')

  Unsorted: each row group contains all 5 categories
  → min='Books', max='Sports' for every row group
  → NO row groups can be skipped


## Step 6 — Parquet vs ORC: Internal Comparison

Both are columnar formats. Key differences:


In [7]:
ORC_PATH = f"{PQ_DIR}/orc_default"
PQ_PATH  = f"{PQ_DIR}/parquet_for_compare"

df.write.mode("overwrite").orc(ORC_PATH)
df.write.mode("overwrite").parquet(PQ_PATH)

def dir_mb(path):
    r = subprocess.run(["du","-sm",path], capture_output=True, text=True)
    return int(r.stdout.split()[0]) if r.stdout else 0

print("Parquet vs ORC comparison:")
print(f"  {'Metric':<35} {'Parquet':>12} {'ORC':>12}")
print("  " + "-"*62)

# Size
pq_mb  = dir_mb(PQ_PATH)
orc_mb = dir_mb(ORC_PATH)
print(f"  {'Storage size (MB)':<35} {pq_mb:>12} {orc_mb:>12}")

# Full scan
runs = 3
for label, pq_path, orc_path in [
    ("Full scan + agg", PQ_PATH, ORC_PATH),
]:
    pq_times, orc_times = [], []
    for _ in range(runs):
        t0 = time.time(); spark.read.parquet(pq_path).agg(F.sum("revenue")).collect()
        pq_times.append(time.time()-t0)
        t0 = time.time(); spark.read.orc(orc_path).agg(F.sum("revenue")).collect()
        orc_times.append(time.time()-t0)
    pq_med  = sorted(pq_times)[1]
    orc_med = sorted(orc_times)[1]
    print(f"  {label:<35} {pq_med:>10.3f}s {orc_med:>10.3f}s")

# Filter
pq_times, orc_times = [], []
for _ in range(runs):
    t0=time.time(); spark.read.parquet(PQ_PATH).filter(F.col("category")=="Electronics").count()
    pq_times.append(time.time()-t0)
    t0=time.time(); spark.read.orc(ORC_PATH).filter(F.col("category")=="Electronics").count()
    orc_times.append(time.time()-t0)
print(f"  {'Selective filter (1 category)':<35} {sorted(pq_times)[1]:>10.3f}s {sorted(orc_times)[1]:>10.3f}s")

print()
print("Parquet: better Spark integration, wider ecosystem (Iceberg, Delta, Hudi)")
print("ORC:     slightly better compression on string-heavy data, Hive-native")

Parquet vs ORC comparison:
  Metric                                   Parquet          ORC
  --------------------------------------------------------------
  Storage size (MB)                              9            4
  Full scan + agg                          0.284s      0.254s
  Selective filter (1 category)            0.287s      0.251s

Parquet: better Spark integration, wider ecosystem (Iceberg, Delta, Hudi)
ORC:     slightly better compression on string-heavy data, Hive-native


## Summary: Parquet Internals Cheat Sheet

### File layout
```
Parquet = [Row Groups] + [File Footer with statistics]
Row Group ≈ 128 MB (tune: spark.hadoop.parquet.block.size)
Column Chunk = one column in one row group
Page ≈ 1 MB (tune: parquet.page.size)
```

### Encodings reference
| Encoding | Used for | Compression ratio |
|---|---|---|
| PLAIN_DICTIONARY | Low-cardinality strings/ints (< 32K unique) | 5-20x |
| RLE | Booleans, repeated values | 2-10x |
| DELTA_BINARY_PACKED | Monotonic integers (IDs, timestamps) | 3-8x |
| PLAIN | Floats, high-cardinality | 1x |

### Performance tuning
```python
# For write-optimised (better compression):
.option("parquet.block.size", str(256*1024*1024))   # 256 MB row groups

# For read-optimised (better skipping):
.option("parquet.block.size", str(32*1024*1024))    # 32 MB row groups

# Sort before write for max data skipping:
df.orderBy("filter_column").write.parquet(path)
```

### Data skipping effectiveness
```
Sorted by filter column  → 80-99% row groups skipped
Random order             → 0% row groups skipped (min/max overlap everywhere)
```
